# Day E2 — 표 구조, 접근, 윤리, 비식별화: MIMIC 계열 데이터 읽는 법

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/E-ehr/day-E2.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/E-ehr/day-E2.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 세 겹의 열쇠로 안전하게 합치기

MIMIC-IV와 같은 열 이름을 쓴 가상 표 세 개를 만들어 합칩니다.

In [ ]:
import sqlite3
import pandas as pd

patients = pd.DataFrame({
    "subject_id": [101, 102, 103, 104],
    "gender": ["F", "M", "F", "M"],
    "anchor_age": [67, 54, 81, 45],
})
admissions = pd.DataFrame({
    "subject_id": [101, 101, 102, 103, 104],
    "hadm_id":    [2001, 2002, 2003, 2004, 2005],
    "admittime": pd.to_datetime(["2150-01-03", "2150-06-20", "2151-03-11",
                                 "2149-11-30", "2152-08-02"]),
})
icustays = pd.DataFrame({
    "subject_id": [101, 101, 101, 102, 103],
    "hadm_id":    [2001, 2001, 2002, 2003, 2004],
    "stay_id":    [3001, 3002, 3003, 3004, 3005],
    "intime": pd.to_datetime(["2150-01-04 02:00", "2150-01-09 15:00", "2150-06-21 08:00",
                              "2151-03-11 20:00", "2149-12-01 06:00"]),
})
for name, df, key in [("patients", patients, "subject_id"),
                      ("admissions", admissions, "hadm_id"),
                      ("icustays", icustays, "stay_id")]:
    print(f"{name:10s} 행 {len(df)}개, 열쇠 {key} 중복 없음: {df[key].is_unique}")

stays = (icustays
         .merge(admissions[["hadm_id", "admittime"]], on="hadm_id", how="left", validate="many_to_one")
         .merge(patients, on="subject_id", how="left", validate="many_to_one"))
print(stays[["subject_id", "hadm_id", "stay_id", "intime", "anchor_age"]])
print("ICU 기록이 없는 환자:", sorted(set(patients.subject_id) - set(icustays.subject_id)))

### 예시 2 — SQL 창 함수와 pandas로 "환자별 첫 ICU 입실" 고르기

같은 노트북에서 예시 1 바로 다음 셀에 실행합니다. `sqlite3`는 Python에 기본으로 들어 있어 따로 설치할 것이 없습니다.

In [ ]:
con = sqlite3.connect(":memory:")
icustays.assign(intime=icustays.intime.astype(str)).to_sql("icustays", con, index=False)
q = """
WITH ranked AS (
  SELECT subject_id, stay_id, intime,
         RANK() OVER (PARTITION BY subject_id ORDER BY intime) AS stay_rank
  FROM icustays
)
SELECT subject_id, stay_id, intime FROM ranked WHERE stay_rank = 1 ORDER BY subject_id;
"""
first_sql = pd.read_sql(q, con)
print(first_sql)
first_pd = icustays.sort_values("intime").groupby("subject_id").head(1).sort_values("subject_id")
print("pandas 결과와 같은가:", first_sql["stay_id"].tolist() == first_pd["stay_id"].tolist())

### 예시 3 — 날짜 이동은 간격을 보존한다

비식별화의 날짜 이동을 흉내 냅니다(이동량은 가상의 값입니다).

In [ ]:
import pandas as pd

true_dates = pd.DataFrame({
    "subject_id": [101, 101, 102],
    "event": ["입원", "퇴원", "입원"],
    "real_time": pd.to_datetime(["2019-02-01 10:00", "2019-02-09 16:00", "2020-07-15 09:00"]),
})
shift_years = {101: 131, 102: 130}           # 환자마다 다른 무작위 이동(가상의 값)
true_dates["shifted"] = true_dates.apply(
    lambda r: r.real_time + pd.DateOffset(years=shift_years[r.subject_id]), axis=1)
p101 = true_dates[true_dates.subject_id == 101]
print(true_dates)
print("101번 환자 재원 기간(실제):", p101.real_time.iloc[1] - p101.real_time.iloc[0])
print("101번 환자 재원 기간(이동 후):", p101.shifted.iloc[1] - p101.shifted.iloc[0])

## 직접 해보기

1. `notebooks/advanced/E/E2_tables_and_access.ipynb`를 만들고 예시 1–3을 실행합니다.
2. 일부러 사고를 내 봅니다. 예시 1의 `admissions`에서 `hadm_id` 2003을 2002로 바꿔(중복 열쇠) 다시 실행하면 `merge`가 어떤 오류를 내는지 읽고, 오류 메시지의 핵심 단어를 노트에 적습니다. 원래대로 돌려놓습니다.
3. (선택, 데모 데이터) [MIMIC-IV Clinical Database Demo v2.2](https://physionet.org/content/mimic-iv-demo/2.2/) 페이지 아래 Files에서 ZIP을 내려받아 **저장소 밖**의 폴더(또는 `.gitignore`에 넣은 `data/` 아래)에 풉니다. 라이선스(`LICENSE.txt`)를 읽고, `SHA256SUMS.txt`로 파일이 온전한지 확인합니다.

In [ ]:
import hashlib
from pathlib import Path

root = Path("../../../data/mimic_iv_demo/mimic-iv-clinical-database-demo-2.2")   # 내가 푼 위치로 바꾼다
bad = 0
for line in (root / "SHA256SUMS.txt").read_text().splitlines():
    h, name = line.split(maxsplit=1)
    if hashlib.sha256((root / name).read_bytes()).hexdigest() != h:
        print("불일치:", name); bad += 1
print("검사한 파일 수:", len(open(root / "SHA256SUMS.txt").readlines()), "| 불일치:", bad)

4. (선택, 계속) 데모의 세 표로 예시 1과 같은 검사를 합니다. **출력은 열 이름과 True/False만** 확인하고, 환자별 값은 노트북 밖으로 옮기지 않습니다.

In [ ]:
import pandas as pd

patients = pd.read_csv(root / "hosp" / "patients.csv.gz")
admissions = pd.read_csv(root / "hosp" / "admissions.csv.gz", parse_dates=["admittime", "dischtime"])
icustays = pd.read_csv(root / "icu" / "icustays.csv.gz", parse_dates=["intime", "outtime"])
for name, df in [("patients", patients), ("admissions", admissions), ("icustays", icustays)]:
    print(name, "| 열:", ", ".join(df.columns[:8]), "...")
print("subject_id 유일:", patients.subject_id.is_unique,
      "| hadm_id 유일:", admissions.hadm_id.is_unique, "| stay_id 유일:", icustays.stay_id.is_unique)
stays = icustays.merge(admissions[["hadm_id", "admittime"]], on="hadm_id", how="left", validate="many_to_one")
print("ICU 입실 시각이 입원 시각보다 앞선 행:", int((stays.intime < stays.admittime).sum()))
first = icustays.sort_values("intime").groupby("subject_id").head(1)
print("첫 ICU 입실만 남기면 행 수가 줄었는가:", len(first) < len(icustays))